In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

In [ ]:
data =pd.read_csv('netflix1.csv')
print(data.head())

In [ ]:
print(data.isnull().sum())
data.drop_duplicates(inplace=True)
data.dropna(subset=['director','country'], inplace=True)
data['date_added'] = pd.to_datetime(data['date_added'])
print(data.dtypes)

In [ ]:
print(data.columns)

In [ ]:
type_counts = data['type'].value_counts()
plt.figure(figsize=(8, 6))
sns.barplot(x=type_counts.index, y=type_counts.values,palette='Set2')
plt.title('Distribution of Content by Type')
plt.xlabel('Type')
plt.ylabel('Count')
plt.show()

In [ ]:
data['genres'] = data['listed_in'].apply(lambda x: x.split(','))
all_genres = sum(data['genres'], [])
genre_counts = pd.Series(all_genres).value_counts().head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=genre_counts.values, y=genre_counts.index,palette='Set3')
plt.title('Most Common Genres on Netflix')
plt.xlabel('Count')
plt.ylabel('Genre')
plt.show()

In [ ]:
data['year_added'] = data['date_added'].dt.year
data['month_added'] = data['date_added'].dt.month

plt.figure(figsize=(12, 6))
sns.countplot(x='year_added', data=data, palette='coolwarm')
plt.title('Content Added Over Time')
plt.xlabel('Year')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
top_directors = data['director'].value_counts().head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_directors.values,y=top_directors.index,palette='Blues_d')
plt.title('Top 10 Directors with the Most Titles')
plt.xlabel('Number of Titles')
plt.ylabel('Director')
plt.show()

In [ ]:
movie_titles = data[data['type'] == 'Movie']['title']
wordcloud=WordCloud(width=800,height=400,background_color='black').generate(' '.join(movie_titles))
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:

data['listed_in'] = data['listed_in'].fillna('Unknown')
data['num_genres'] = data['listed_in'].apply(
    lambda x: len([g.strip() for g in str(x).split(',') if g.strip()])
)

def extract_minutes(duration, content_type):
    if pd.isna(duration):
        return np.nan
    s = str(duration).lower().strip()
    if 'min' in s:
        try:
            return int(''.join(ch for ch in s if ch.isdigit()))
        except:
            import re
            match = re.search(r'(\d+)', s)
            return int(match.group(1)) if match else np.nan
    return np.nan

data['duration_minutes'] = data.apply(
    lambda row: extract_minutes(row['duration'], row['type']), axis=1
)

data['is_movie'] = data['type'].str.lower().apply(lambda x: 1 if x == 'movie' else 0)

max_year = int(data['release_year'].dropna().max())
data['release_age'] = max_year - data['release_year']

data['title_len'] = data['title'].fillna('').apply(lambda x: len(str(x).split()))

data[['title', 'type', 'num_genres', 'duration_minutes', 'is_movie', 'release_year', 'release_age', 'title_len']].head()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity
from difflib import get_close_matches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import re

# Load data if not available in environment
try:
    if 'data' not in globals() or 'text_features' not in data.columns:
        print("Required data missing. Loading and cleaning data...")
        data = pd.read_csv('netflix1.csv')
        data.drop_duplicates(inplace=True)
        data.dropna(subset=['director', 'country'], inplace=True)
        data.reset_index(drop=True, inplace=True)
        
        data['listed_in'] = data['listed_in'].fillna('Unknown')
        data['year_added'] = pd.to_datetime(data['date_added']).dt.year
        data['month_added'] = pd.to_datetime(data['date_added']).dt.month
        
        # Derived features
        data['num_genres'] = data['listed_in'].apply(lambda x: len([g.strip() for g in str(x).split(',') if g.strip()]))
        
        def extract_minutes(duration):
            if pd.isna(duration):
                return np.nan
            s = str(duration).lower().strip()
            if 'min' in s:
                try:
                    match = re.search(r'(\d+)', s)
                    return int(match.group(1)) if match else np.nan
                except:
                    return np.nan
            return np.nan
            
        data['duration_minutes'] = data['duration'].apply(extract_minutes)
        data['duration_minutes'].fillna(data['duration_minutes'].median(), inplace=True)
        
        data['is_movie'] = data['type'].str.lower().apply(lambda x: 1 if x == 'movie' else 0)
        max_year = int(data['release_year'].dropna().max())
        data['release_age'] = max_year - data['release_year']
        data['release_age'].fillna(data['release_age'].median(), inplace=True)
        data['title_len'] = data['title'].fillna('').apply(lambda x: len(str(x).split()))
        
        print("Data loaded successfully.")
except Exception as e:
    print(f"Error loading data: {e}")

# Data Cleaning for better NLP recommendations
def clean_text_data(x):
    if isinstance(x, str):
        # Remove spaces and commas to create unique tokens (e.g. "David Fincher" -> "davidfincher")
        return x.replace(' ', '').replace(',', ' ').lower()
    return ''

# Create cleaned feature strings
data['director_clean'] = data['director'].apply(clean_text_data)
data['listed_in_clean'] = data['listed_in'].apply(clean_text_data)
data['country_clean'] = data['country'].apply(clean_text_data)

# Improved Feature Engineering:
# Weighting genres (3x) and directors (2x) more heavily.
data['text_features'] = (data['title'].fillna('').str.lower() + ' ') + \
                        (data['listed_in_clean'] + ' ') * 3 + \
                        (data['director_clean'] + ' ') * 2 + \
                        (data['country_clean'] + ' ') + \
                        data['rating'].fillna('').str.lower()

# Vectorization using TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['text_features'])

# Numeric Feature Scaling
numeric_features = data[['duration_minutes', 'num_genres', 'is_movie', 'release_age', 'title_len']].fillna(0)
scaler = MinMaxScaler()
numeric_scaled = scaler.fit_transform(numeric_features)

# Similarity Calculation
text_similarity = linear_kernel(tfidf_matrix, tfidf_matrix)
numeric_similarity = cosine_similarity(numeric_scaled, numeric_scaled)

# Hybrid Model: 85% text based, 15% numeric based
hybrid_similarity = 0.85 * text_similarity + 0.15 * numeric_similarity

def find_closest_title(user_input, df):
    # Case insensitive matching
    titles_lower = [str(t).lower() for t in df['title'].values]
    matches = get_close_matches(user_input.lower(), titles_lower, n=1, cutoff=0.5)
    if not matches:
        return None
    matched_idx = titles_lower.index(matches[0])
    return df['title'].iloc[matched_idx]

def get_recommendations(title, df, sim_matrix, top_n=5):
    idx = np.where(df['title'] == title)[0][0]
    sim_scores = list(enumerate(sim_matrix[idx]))
    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # Get top_n ignoring the first one (itself)
    sim_scores = sim_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    return df.iloc[movie_indices][['title', 'type', 'listed_in', 'director', 'country', 'rating']]

def get_content_trends(title, df, sim_matrix, top_n=5):
    idx = np.where(df['title'] == title)[0][0]
    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    
    subset = df.iloc[movie_indices]
    trends = {
        'Average Duration': f"{subset['duration_minutes'].mean():.1f} mins",
        'Top Genres': ', '.join(subset['listed_in'].value_counts().head(3).index.tolist()),
        'Top Directors': ', '.join(subset['director'].value_counts().head(2).index.tolist()),
        'Top Countries': ', '.join(subset['country'].value_counts().head(2).index.tolist())
    }
    return trends

# --------------- Widget UI ---------------
title_input= widgets.Text(
    value='',
    placeholder='E.g., Inception, Breaking Bad...',
    description='Search:',
    layout=widgets.Layout(width='400px', margin='0 0 10px 0')
)

output_area = widgets.Output()

def on_search(change):
    with output_area:
        clear_output()
        user_text = change['new'].strip()
        
        if not user_text:
            return
            
        if user_text.lower() == 'exit':
            display(HTML("<h3>Goodbye! 👋</h3>"))
            return
            
        matched_title = find_closest_title(user_text, data)
        if not matched_title:
            display(HTML(f"<div style='color:#e50914; font-weight:bold;'>No close match found for '{user_text}'. Please try another title.</div>"))
            return
            
        display(HTML(f"<h3 style='color:#22bb33;'>🎬 Closest Match: {matched_title}</h3>"))
        
        recommendations = get_recommendations(matched_title, data, hybrid_similarity)
        display(HTML("<h4>🌟 Top Recommendations:</h4>"))
        
        # Build pretty table for recommendations
        table_html = "<table style='width:100%; border-collapse: collapse;'>"
        table_html += "<tr style='background-color:#f1f1f1;'><th style='padding:8px;text-align:left;'>Title</th><th>Type</th><th>Genres</th><th>Director</th></tr>"
        for _, row in recommendations.iterrows():
            table_html += f"<tr><td style='padding:8px; border-bottom:1px solid #ddd;'><b>{row['title']}</b></td>"
            table_html += f"<td style='padding:8px; border-bottom:1px solid #ddd;'>{row['type']}</td>"
            table_html += f"<td style='padding:8px; border-bottom:1px solid #ddd;'>{row['listed_in']}</td>"
            table_html += f"<td style='padding:8px; border-bottom:1px solid #ddd;'>{row['director']}</td></tr>"
        table_html += "</table><br>"
        display(HTML(table_html))
        
        # Display Trends
        display(HTML("<h4>📈 Insights from Similar Titles:</h4>"))
        trends = get_content_trends(matched_title, data, hybrid_similarity)
        list_html = "<ul>"
        for k, v in trends.items():
            if pd.notna(v) and v != '':
                list_html += f"<li><b>{k}:</b> {v}</li>"
        list_html += "</ul>"
        display(HTML(list_html))

title_input.observe(on_search, names='value')

display(HTML("<h2>🎥 Netflix Recommendation System</h2>"))
display(widgets.VBox([title_input, output_area]))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data=pd.read_csv("netflix1.csv")
data.head()


In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data=data.drop_duplicates()

In [ ]:
data['type'].value_counts()

In [ ]:
freq=data['type'].value_counts()
fig,axes=plt.subplots(1,2,figsize=(8,4))
sns.countplot(data,x=data['type'],ax=axes[0])
plt.pie(freq,labels=['Movie','TVShow'],autopct='%.0f%%')
plt.suptitle('TotalContentonNetflix',fontsize=20)

In [ ]:
data['rating'].value_counts()

In [ ]:
ratings=data['rating'].value_counts().reset_index().sort_values(by='count', ascending=False)
plt.bar(ratings['rating'], ratings['count'])
plt.xticks(rotation=45, ha='right')
plt.xlabel("Rating Types")
plt.ylabel("Rating Frequency")
plt.suptitle('Rating on Netflix', fontsize=20)

In [ ]:
plt.pie(ratings['count'][:8], labels=ratings['rating'][:8],
autopct='%.0f%%')
plt.suptitle('Rating on Netflix', fontsize=20)

In [ ]:
data['date_added']=pd.to_datetime(data['date_added'])

In [ ]:
data.describe()

In [ ]:
data['country'].value_counts()

In [ ]:
top_ten_countries=data['country'].value_counts().reset_index().sort_values(by='count',ascending=False)[:10]
plt.figure(figsize=(10,6))
plt.bar(top_ten_countries['country'],
top_ten_countries['count'])
plt.xticks(rotation=45,ha='right')
plt.xlabel("Country")
plt.ylabel("Frequency")
plt.suptitle("Top10countrieswithmostcontentonNetflix")
plt.show()


In [ ]:
data['year']=data['date_added'].dt.year
data['month']=data['date_added'].dt.month
data['day']=data['date_added'].dt.day

In [ ]:
monthly_movie_release=data[data['type']=='Movie']['month'].value_counts().sort_index()
monthly_series_release=data[data['type']=='TVShow']['month'].value_counts().sort_index()
plt.plot(monthly_movie_release.index,monthly_movie_release.values,label='Movies')
plt.plot(monthly_series_release.index,monthly_series_release.values,label='Series')
plt.xlabel("Months")
plt.ylabel("Frequencyofreleases")
plt.xticks(range(1,13),['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.legend()
plt.grid(True)
plt.suptitle("MonthlyreleasesofMoviesandTVshowsonNetflix")
plt.show()


In [ ]:
yearly_movie_releases=data[data['type']=='Movie']['year'].value_counts().sort_index()
yearly_series_releases=data[data['type']=='TVShow']['year'].value_counts().sort_index()
plt.plot(yearly_movie_releases.index,yearly_movie_releases.values, label='Movies')
plt.plot(yearly_series_releases.index,yearly_series_releases.values, label='TV Shows')
plt.xlabel("Years")
plt.ylabel("Frequency of releases")
plt.grid(True)
plt.suptitle("Yearly releases of Movies and TV Shows onNetflix")
plt.legend()

In [ ]:
popular_movie_genre=data[data['type']=='Movie'].groupby("listed_in").size().sort_values(ascending=False)[:10]
popular_series_genre=data[data['type']=='TVShow'].groupby("listed_in").size().sort_values(ascending=False)[:10]
plt.bar(popular_movie_genre.index, popular_movie_genre.values)
plt.xticks(rotation=45, ha='right')
plt.xlabel("Genres")
plt.ylabel("Movies Frequency")
plt.suptitle("Top 10 popular genres for movies on Netflix")
plt.show()

In [ ]:
plt.bar(popular_series_genre.index,popular_series_genre.values)
plt.xticks(rotation=45, ha='right')
plt.xlabel("Genres")
plt.ylabel("TV Shows Frequency")
plt.suptitle("Top 10 popular genres for TV Shows on Netflix")
plt.show()

In [ ]:
directors=data['director'].value_counts().reset_index().sort_values(by='count', ascending=False)[1:15]
plt.bar(directors['director'], directors['count'])
plt.xticks(rotation=45, ha='right')